# Fase 1 — Da Ideia ao MVP: uma Chain Executável com Entrada e Saída Estruturadas

**Projeto:** Tech Lead Agêntico, o primeiro agente de uma *fábrica agêntica de desenvolvimento de software* · **Aluno:** Adriano Carvalho · MBA em Engenharia de IA (Infnet)

Nesta fase construímos a menor chain que já produz valor: o agente recebe uma **demanda em linguagem natural** e devolve uma **triagem técnica em JSON** (tipo, projeto, componentes, complexidade, riscos, perguntas em aberto e próximos passos). Antes do código, descrevemos a anatomia e a arquitetura do agente que será construído até a Fase 5.

> Este notebook roda em dois modos: **real** (Claude, com `ANTHROPIC_API_KEY` no `.env`) ou **demonstração** (`LLM_PROVIDER=demo`, sem chave e sem custo). A primeira célula mostra o modo em uso.

In [1]:
import sys, pathlib
RAIZ = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(RAIZ))

from fabrica_agentica.config import descrever_ambiente
print(descrever_ambiente())

Provedor de LLM : anthropic
APIs externas   : rede (ao vivo)
Embeddings      : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Modelo NLI      : MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli
LangSmith       : ligado
Repositório GH  : DriccoC/fabrica-agentica-sandbox


## 1. Definição do problema e domínio

Uma fábrica de software recebe demandas vagas ("quero login com Google no Portal") e precisa transformá-las em trabalho planejável: histórias com critérios de aceite, estimativas compatíveis com a capacidade do time, bibliotecas seguras e aderência às normas internas. Hoje isso consome horas do líder técnico, que consulta o backlog, o manual de engenharia, o PyPI, bases de vulnerabilidades e o GitHub.

O **Tech Lead Agêntico** automatiza esse trabalho para a NovaForja Software (empresa fictícia, com os projetos Portal, PagaFacil e AppCampo). Nesta disciplina ele é um **agente único**; na próxima disciplina do bloco ele será o orquestrador de outros agentes da fábrica (arquiteto, desenvolvedor, QA, segurança).

**Por que um agente, e não só uma chain?** A triagem desta fase é uma chain: entrada e saída previsíveis, passos fixos. Mas planejar uma demanda exige dados que mudam com o tempo (capacidade da sprint, feriados, versões e vulnerabilidades de bibliotecas, issues abertas) e decisões que dependem do caso (consultar ou não uma norma, checar uma biblioteca). É esse o ponto em que a chain deixa de bastar e o ciclo de raciocínio e ação se justifica.

## 2. Anatomia do agente

| Componente | No Tech Lead Agêntico |
|---|---|
| **LLM (cérebro)** | Claude (Anthropic) via `langchain-anthropic`: `claude-sonnet-5-5` para raciocínio e `claude-haiku-5-5` para tarefas auxiliares (extração, resumo); OpenAI como alternativa configurável |
| **System message** | prompt PTCF (Persona, Tarefa, Contexto, Formato) com regras de raciocínio, *few-shot* e *chain-of-thought* (Fase 2) |
| **Memória de curto prazo** | a conversa da sessão: demanda em discussão, projeto, sprint e pessoas citadas (buffer com janela e sumarização) |
| **Memória de longo prazo** | semântica: stack, convenções e preferências por projeto; episódica: demandas e decisões anteriores; procedural: Definição de Pronto e fluxo de trabalho (no system prompt) |
| **Ferramentas** | hardcoded (capacidade da sprint, semver, faixa CVSS), semânticas (decompor demanda, resumir), APIs externas (BrasilAPI, PyPI, OSV.dev, GitHub), *text-to-query* (backlog SQL) e ação com aprovação humana (registrar histórias e abrir issues) |
| **Base de conhecimento** | Manual de Engenharia (normas MAN-01 a MAN-07) num índice FAISS já existente |
| **Verificação** | camada de V&V: fact-checking determinístico + modelo NLI + coerência lógica (Fase 5) |

## 3. Classificação arquitetural e maturidade agêntica

- **Padrão: híbrido.** É *deliberativo* ao planejar (decompõe a demanda em etapas antes de agir, compara o plano com restrições) e *reativo* ao executar (ciclo ReAct: a cada observação de ferramenta decide o próximo passo). Um agente puramente reativo responderia bem a perguntas pontuais, mas não sustentaria o planejamento de uma demanda inteira; um puramente deliberativo seria rígido demais para conversas.
- **Espectro de maturidade:**

| Nível | Descrição | Onde o projeto está |
|---|---|---|
| 1. Chain fixa | passos definidos no código, saída estruturada | **hoje (Fase 1)** |
| 2. Agente com ferramentas | decide quando e qual ferramenta usar (ReAct) | Fase 2 |
| 3. Agente com memória e auto-verificação | contexto entre turnos e sessões, checa as próprias respostas | **fim do projeto (Fases 3 a 5)** |
| 4. Sistema multiagente | agentes especialistas coordenados | próxima disciplina |

## 4. Arquitetura de planejamento (decomposição hierárquica)

Para demandas complexas, o agente decompõe a tarefa em níveis (alinhado ao MAN-06 §5):

```
Demanda ("login com Google no Portal na próxima sprint")
└── Objetivo: plano de trabalho verificado
    ├── 1. Restrições do time  -> capacidade da sprint (SQL + feriados + cálculo)
    ├── 2. Restrições normativas -> MAN-05 (autenticação), ADR-012, MAN-01 (DoD)
    ├── 3. Restrições técnicas  -> versão e CVEs da biblioteca (PyPI + OSV)
    ├── 4. Contexto existente   -> issues relacionadas (GitHub)
    ├── 5. Decomposição          -> épico → histórias (Gherkin, Fibonacci) → tarefas
    ├── 6. Validação             -> total de pontos ≤ saldo da sprint? histórias ≤ 8 pontos?
    └── 7. Ação (com aprovação)  -> registrar no backlog e abrir issues
```

## 5. Arquitetura de memória aumentada (desenho; implementação na Fase 3)

| Tipo (CoALA) | Horizonte | Exemplo no domínio | Onde vive |
|---|---|---|---|
| Trabalho (curto prazo) | a sessão | "estamos falando do Portal, sprint 13; tirar o Bruno" | histórico de mensagens com janela + resumo |
| Semântica (longo prazo) | permanente | "o Portal usa FastAPI e PostgreSQL", "o usuário prefere histórias de até 5 pontos" | SQLite + busca vetorial |
| Episódica (longo prazo) | permanente | "em 05/11 planejamos o login com Google: 13 pontos" | SQLite + busca vetorial |
| Procedural | permanente | Definição de Pronto, regras de capacidade | system prompt e ferramentas hardcoded |

## 6. Templates de prompt reutilizáveis

- `PromptTemplate`: chamada de texto simples (um bloco de texto, sem papéis) — usado no resumo de uma linha para o quadro de tarefas.
- `ChatPromptTemplate`: formato de mensagens (`system` + `human`) de um modelo de chat — usado na triagem.

Ambos ficam em `fabrica_agentica/prompts.py` para serem reutilizados nas próximas fases.

In [2]:
from fabrica_agentica.prompts import PROMPT_RESUMO, PROMPT_ANALISE

demanda = "Quero login com Google no projeto Portal na próxima sprint."

print("PromptTemplate — variáveis:", PROMPT_RESUMO.input_variables)
print(PROMPT_RESUMO.format(demanda=demanda))
print("\nChatPromptTemplate — variáveis:", PROMPT_ANALISE.input_variables)
for m in PROMPT_ANALISE.format_messages(demanda=demanda, instrucoes_formato="(instruções de formato)"):
    print(f"[{m.type}] {m.content[:160]}...")

PromptTemplate — variáveis: ['demanda']
Resuma, em uma única frase objetiva para um quadro de tarefas, a seguinte demanda de software:
Quero login com Google no projeto Portal na próxima sprint.

Resumo:

ChatPromptTemplate — variáveis: ['demanda', 'instrucoes_formato']
[system] Você é o líder técnico da NovaForja Software, empresa (fictícia) com os projetos Portal (Python/FastAPI + React), PagaFacil (Java/Quarkus) e AppCampo (Flutter)....
[human] DEMANDA: Quero login com Google no projeto Portal na próxima sprint....


## 7. Saída estruturada em JSON com *output parser*

O `PydanticOutputParser` gera as **instruções de formato** (um JSON Schema derivado da classe `AnaliseDemanda`) que são injetadas no prompt, e depois **valida** a resposta do modelo, devolvendo um objeto Python tipado. Assim o resto do sistema não precisa interpretar texto livre.

In [3]:
from fabrica_agentica.chains.fase1 import parser_analise
from fabrica_agentica.esquemas import AnaliseDemanda

print(parser_analise.get_format_instructions()[:700], "...")

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"description": "Saída da chain MVP (Fase 1): a triagem técnica de uma demanda.", "properties": {"resumo": {"description": "Resumo técnico da demanda em uma ou duas frases", "title": "Resumo", "type": "string"}, "tipo": {"description": "Tipo de dema ...


## 8. Composição declarativa (LCEL) e a interface `Runnable`

A chain declarativa é `prompt | modelo | parser`. Todo componente LCEL implementa a interface `Runnable`: `invoke` (uma chamada), `batch` (várias entradas, em paralelo) e `stream` (resposta token a token).

In [4]:
import json
from fabrica_agentica.chains.fase1 import chain_analise_lcel, chain_resumo

chain_lcel = chain_analise_lcel()
resultado = chain_lcel.invoke({"demanda": demanda})          # invoke: uma chamada
print(type(resultado).__name__)
print(json.dumps(resultado.model_dump(), ensure_ascii=False, indent=2))

AnaliseDemanda
{
  "resumo": "Implementar autenticação via Google (OAuth 2.0 / OpenID Connect) no Portal, permitindo que usuários façam login com a conta Google, integrando o fluxo ao backend FastAPI e à interface React, com entrega prevista para a próxima sprint.",
  "tipo": "nova_funcionalidade",
  "projeto": "Portal",
  "componentes_afetados": [
    "backend (FastAPI): endpoints de callback OAuth, validação do ID token e emissão de sessão/JWT",
    "front-end (React): botão 'Entrar com Google' e tratamento do redirecionamento e de erros",
    "banco de dados: vínculo entre usuário e identidade Google (ex.: google_sub), possível migração de schema",
    "configuração e segredos: Client ID/Secret no Google Cloud Console, variáveis de ambiente por ambiente",
    "modelo de usuários e autorização: associação com contas existentes e perfis/permissões",
    "testes automatizados e documentação"
  ],
  "complexidade": "media",
  "riscos": [
    "Duplicidade ou conflito de contas se o e-mai

In [5]:
demandas = [
    {"demanda": "Quero login com Google no projeto Portal na próxima sprint."},
    {"demanda": "O PagaFacil está duplicando a conciliação de boletos no fim do mês, precisamos corrigir."},
    {"demanda": "Qual a melhor forma de adicionar modo escuro no AppCampo?"},
]
for r in chain_lcel.batch(demandas, config={"max_concurrency": 3}):   # batch: lote em paralelo
    print(f"{r.tipo:20} | {r.projeto or '-':10} | {r.complexidade:6} | {r.resumo[:70]}")

nova_funcionalidade  | Portal     | media  | Implementar autenticação via Google (OAuth 2.0 / OpenID Connect) no Po
correcao             | PagaFacil  | media  | O PagaFacil (Java/Quarkus) está executando ou registrando em duplicida
duvida               | AppCampo   | media  | Pergunta sobre a melhor abordagem para implementar modo escuro (dark m


In [6]:
# stream: a resposta chega em pedaços (útil para chat); aqui com a chain de resumo (texto livre)
for pedaco in chain_resumo().stream({"demanda": demanda}):
    print(pedaco, end="|", flush=True)

|

Implementar aut|

enticação com|

 Google (|

login social|

) no projeto Portal|

 na|

 próxima sprint.|

|

## 9. Composição imperativa com `@chain`

A mesma triagem como uma função Python decorada com `@chain`. A função vira um `Runnable` (tem `invoke`, `batch`, `stream`), mas o corpo é código comum, com controle total: aqui, se o JSON vier inválido, a função devolve o erro ao modelo e pede uma nova resposta — algo que não cabe num pipe simples.

In [7]:
import inspect
from fabrica_agentica.chains.fase1 import analisar_demanda_imperativa

print(inspect.getsource(analisar_demanda_imperativa.func))
r = analisar_demanda_imperativa.invoke(demanda)
print(type(r).__name__, "|", r.tipo, "|", r.complexidade, "|", r.componentes_afetados)

@chain
def analisar_demanda_imperativa(demanda: str) -> AnaliseDemanda:
    """Composição imperativa: mesma lógica, com uma nova tentativa se o JSON vier inválido."""
    llm = criar_llm("agente")
    mensagens = prompt_analise.format_messages(demanda=demanda)
    resposta = llm.invoke(mensagens)
    try:
        return parser_analise.parse(texto_de(resposta))
    except OutputParserException as erro:
        correcao = HumanMessage(
            content=f"Sua resposta não é um JSON válido para o esquema ({erro}). Responda novamente apenas com o JSON."
        )
        resposta = llm.invoke([*mensagens, resposta, correcao])
        return parser_analise.parse(texto_de(resposta))



AnaliseDemanda | nova_funcionalidade | media | ['backend (FastAPI): endpoints de autenticação, callback OAuth e validação do ID token', "front-end (React): botão 'Entrar com Google' e tratamento do fluxo de redirecionamento/estado de sessão", 'banco de dados: vínculo entre usuário e identidade externa (ex.: google_sub), possível migração de schema', 'gestão de sessão/tokens (JWT ou cookies) e autorização', 'configuração e segredos: client ID/secret do Google Cloud Console por ambiente', 'testes automatizados e documentação']


### Teste do tratamento de erro do parser

Simulamos uma resposta malformada do modelo para mostrar o que o parser faz: ele lança `OutputParserException`, que a versão imperativa captura para pedir correção.

In [8]:
from langchain_core.exceptions import OutputParserException
try:
    parser_analise.parse('{"resumo": "faltam campos"}')
except OutputParserException as erro:
    print("OutputParserException capturada:", str(erro)[:200])

OutputParserException capturada: Failed to parse AnaliseDemanda from completion {"resumo": "faltam campos"}. Got: 6 validation errors for AnaliseDemanda
tipo
  Field required [type=missing, input_value={'resumo': 'faltam campos'}, in


## 10. Decisão: qual composição adotar daqui em diante

| Critério | Declarativa (LCEL) | Imperativa (`@chain`) |
|---|---|---|
| `batch`, `stream` e `async` | nativos e otimizados em cada etapa | herdados, mas a função é uma "caixa preta" única |
| Rastreamento no LangSmith | uma etapa por componente (prompt, modelo, parser) | um único passo com o que a função fizer |
| Lógica condicional, laços, novas tentativas | limitada (exige `RunnableBranch`/`RunnableLambda`) | total, é Python comum |
| Composição com outras chains | direta, pelo operador `\|` | possível, mas menos legível |
| Legibilidade para quem revisa | o fluxo é a própria expressão | depende da função |

**Escolha: LCEL como padrão de composição do agente.** Justificativa técnica: (1) o projeto será avaliado com LangSmith, e a granularidade de rastreamento por etapa do LCEL é o que permite diagnosticar onde uma resposta falhou; (2) `batch` paralelo é usado na avaliação em lote e `stream` na interface Streamlit; (3) as chains das próximas fases (sequenciais, paralelas, roteamento) se compõem naturalmente com `|` e `RunnableParallel`. O `@chain` será usado apenas onde houver lógica de controle que não cabe num pipe — como a nova tentativa acima e a camada de verificação da Fase 5.

In [9]:
import time
inicio = time.perf_counter(); chain_lcel.invoke({"demanda": demanda}); t_lcel = time.perf_counter() - inicio
inicio = time.perf_counter(); analisar_demanda_imperativa.invoke(demanda); t_imp = time.perf_counter() - inicio
print(f"LCEL: {t_lcel:.2f}s | @chain: {t_imp:.2f}s  (mesma saída tipada AnaliseDemanda nas duas)")

LCEL: 9.60s | @chain: 11.28s  (mesma saída tipada AnaliseDemanda nas duas)
